# replace-final-head — ex1: swap the final classifier of a toy backbone

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `replace-final-head`. Running the final beacon cell reports progress against the `Transfer: Replace final head` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Transfer: Replace final head` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`replace-final-head`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "replace-final-head"
DD_SUBTOPIC = "Transfer: Replace final head"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Replace final head (transfer learning) — quick refresher

The transfer-learning recipe: take a model pretrained on a big task (e.g. ImageNet, 1000 classes), and swap its final linear classifier for a new head matching your task:

```
import torchvision.models as M
model = M.resnet18(weights='DEFAULT')
in_features = model.fc.in_features        # 512 for resnet18
model.fc = nn.Linear(in_features, num_classes)   # new head, new shape
```

**Why `model.fc.in_features` first.** Different pretrained backbones have different feature widths — ResNet-18/34 have 512, ResNet-50+ have 2048, ViT-B/16 has 768. Read the dim off the OLD head; the new head's `in_features` must match what the backbone produces.

**Why a brand-new `nn.Linear` works.** Setting `model.fc = nn.Linear(...)` REPLACES the attribute on the module. PyTorch's `__setattr__` notices it's an `nn.Module` and registers it as a child — `parameters()`, `state_dict()`, and `.to(device)` all pick it up automatically.

**New head defaults to trainable.** Brand-new `nn.Linear` modules have `requires_grad=True` on their weight + bias. So if you've already frozen the backbone with `requires_grad = False`, the swapped head naturally becomes the ONLY trainable submodule. (See the `freeze-requires-grad` atom for the freeze step.)

**Common pitfall — different head names.** Not every torchvision model calls its head `fc`. Common variants: `model.classifier` (VGG, AlexNet, MobileNet), `model.heads.head` (ViT). Check the model's `repr()` first or inspect `list(model.named_children())[-1]` to find the right attribute name.

### Exercise 1 — swap the final classifier of a toy backbone

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the transfer-learning head-swap pattern: read `model.fc.in_features` to learn the backbone width, then assign `model.fc = nn.Linear(in_features, new_num_classes)`.
> Keywords: transfer-learning, head-replace, in_features, model-fc
> ```

**KCs targeted:** `read-in-features-from-old-head`, `assign-new-linear-to-fc`

A toy pretrained model is provided:

```
class ToyPretrained(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 16),     # backbone output width = 16
        )
        self.fc = nn.Linear(16, 1000)   # ImageNet-style 1000-class head
    def forward(self, x):
        return self.fc(self.backbone(x))
```

Implement `ex1_replace_head(model, new_num_classes)` that performs the head swap, then returns the modified `model`:

1. **Read the backbone width** from the OLD head:
   ```
   in_features = model.fc.in_features
   ```
2. **Replace `model.fc`** with a new `nn.Linear`:
   ```
   model.fc = nn.Linear(in_features, new_num_classes)
   ```
3. Return `model`.

Do NOT touch the backbone — this drill is specifically about the head-swap (the freeze step is a separate atom).

**What the test checks.**
- After the call, `model.fc.out_features == new_num_classes`.
- `model.fc.in_features` STILL equals 16 (the backbone width).
- The new head's params have `requires_grad == True` (brand-new Linear is trainable by default).
- The new head is REGISTERED as a child module (it appears in `model.named_modules()`, `state_dict()`, and `model.parameters()`).
- Forward pass works on a `(B, 10)` input and produces `(B, new_num_classes)`.
- The OLD head's parameters are NO LONGER in `model.parameters()` (replacement is total, not additive).

In [ ]:
import torch.nn as nn

class ToyPretrained(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
        )
        self.fc = nn.Linear(16, 1000)
    def forward(self, x):
        return self.fc(self.backbone(x))


def ex1_replace_head(model, new_num_classes: int):
    """Swap model.fc to an nn.Linear(in_features, new_num_classes). Returns model."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn as nn

    model = ToyPretrained()
    # Snapshot the old head's parameter ids — they must NOT appear in model.parameters() after swap.
    old_head_ids = {id(p) for p in model.fc.parameters()}
    assert model.fc.out_features == 1000, 'fresh ToyPretrained should start with 1000-class head'

    n_classes = 7
    out_model = ex1_replace_head(model, n_classes)
    assert out_model is not None, 'must return the model'

    # Head shape after swap.
    assert out_model.fc.in_features  == 16, f'in_features changed unexpectedly: {out_model.fc.in_features}'
    assert out_model.fc.out_features == n_classes, (
        f'out_features wrong: {out_model.fc.out_features} expected {n_classes}'
    )

    # New head is trainable.
    assert out_model.fc.weight.requires_grad is True
    assert out_model.fc.bias.requires_grad   is True

    # New head registered as a child — appears in state_dict and parameters().
    sd_keys = set(out_model.state_dict().keys())
    assert 'fc.weight' in sd_keys, f'fc.weight missing from state_dict: {sd_keys}'
    assert 'fc.bias'   in sd_keys, f'fc.bias missing from state_dict: {sd_keys}'
    param_ids = {id(p) for p in out_model.parameters()}
    assert id(out_model.fc.weight) in param_ids
    assert id(out_model.fc.bias)   in param_ids

    # Old head's params are GONE — replacement is total, not additive.
    assert old_head_ids.isdisjoint(param_ids), 'old head params still appear in model.parameters() after swap'

    # Forward pass works.
    x = t.randn(4, 10)
    y = out_model(x)
    assert y.shape == (4, n_classes), f'forward output shape wrong: {tuple(y.shape)}'

    # Different sizes round-trip cleanly.
    for n in [1, 2, 5, 100, 10000]:
        m = ToyPretrained()
        m = ex1_replace_head(m, n)
        assert m.fc.out_features == n
        assert m(t.randn(1, 10)).shape == (1, n)

    # The backbone is UNTOUCHED (this drill is only about the head).
    model_fresh = ToyPretrained()
    bb_before = [p.detach().clone() for p in model_fresh.backbone.parameters()]
    model_fresh = ex1_replace_head(model_fresh, 3)
    for p_now, p_then in zip(model_fresh.backbone.parameters(), bb_before):
        assert t.equal(p_now, p_then), 'backbone was modified during head swap'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_replace_head(model, new_num_classes: int):
    import torch.nn as nn
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, new_num_classes)
    return model
```

**Why `model.fc.in_features` (not hard-coding 16).** Different backbones have different feature widths: ResNet-18/34 = 512, ResNet-50+ = 2048, ViT-B/16 = 768, MobileNet-v2 = 1280. Reading the dim off the OLD head makes the swap function backbone-agnostic — same code works on any standard architecture.

**Why `model.fc = ...` works as a registration.** PyTorch's `nn.Module.__setattr__` intercepts assignment of `nn.Module` (or `nn.Parameter`) instances and registers them as children. So a plain Python attribute assignment is enough — no explicit `register_module` call needed. (The same machinery deregisters the old `fc` from `_modules` first.)

**Common pitfall — different head names.** Not every torchvision model calls its head `fc`. Check the model's repr or `list(model.named_children())[-1]` to find the right attribute. Examples:
- ResNet, GoogLeNet → `model.fc`
- VGG, AlexNet, MobileNet → `model.classifier[-1]` (a Sequential)
- ViT → `model.heads.head`
- EfficientNet → `model.classifier[1]`

**Why this composes with `freeze-requires-grad`.** The swap creates a brand-new Linear with `requires_grad=True`. If the backbone is frozen FIRST and the swap happens SECOND, the new head naturally becomes the only trainable submodule. That's the transfer-learning recipe.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()